# Feature Engineering

In the previous notebook, we removed all of the columns that contained redundant information, weren't useful for modeling, required too much processing to make useful, or leaked information from the future. Now we can handle missing values and prepare features for our machine learning algorithms.

In [2]:
## Read in libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
df = pd.read_csv('loans_clean.csv')
df.head()

,loan_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,...,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,last_credit_pull_d,pub_rec_bankruptcies
0,5000.0,36 months,10.65%,162.87,10+ years,RENT,24000.0,Verified,1,credit_card,...,0.0,Jan-1985,1.0,3.0,0.0,13648.0,83.7%,9.0,Jun-2016,0.0
1,2500.0,60 months,15.27%,59.83,< 1 year,RENT,30000.0,Source Verified,0,car,...,0.0,Apr-1999,5.0,3.0,0.0,1687.0,9.4%,4.0,Sep-2013,0.0
2,2400.0,36 months,15.96%,84.33,10+ years,RENT,12252.0,Not Verified,1,small_business,...,0.0,Nov-2001,2.0,2.0,0.0,2956.0,98.5%,10.0,Jun-2016,0.0
3,10000.0,36 months,13.49%,339.31,10+ years,RENT,49200.0,Source Verified,1,other,...,0.0,Feb-1996,1.0,10.0,0.0,5598.0,21%,37.0,Apr-2016,0.0
4,5000.0,36 months,7.90%,156.46,3 years,RENT,36000.0,Source Verified,1,wedding,...,0.0,Nov-2004,3.0,9.0,0.0,7963.0,28.3%,12.0,Jan-2016,0.0


In [4]:
## Find fields with null values
null = df.isnull().sum()
null[null!=0]

emp_length              1036
title                     11
revol_util                50
last_credit_pull_d         2
pub_rec_bankruptcies     697
dtype: int64

Three columns contain relatively few missing values while ```emp_length``` and ```pub_rec_bankruptcies``` contain a relatively high amount of missing values.

Domain knowledge tells us that employment length is frequently used in assessing how risky a potential borrower is, so we'll keep this column despite its relatively large amount of missing values.

Perhaps we can drop ```pub_rec_bankruptcies``` all together...

In [5]:
df['pub_rec_bankruptcies'].value_counts(normalize=True,dropna=False)

0.0    0.939438
1.0    0.042456
NaN    0.017978
2.0    0.000129
Name: pub_rec_bankruptcies, dtype: float64

This column offers very little variability on account of almost 94% of values are in the same category. It makes sense to toss this one out.

In [6]:
## Drop unwanted column and nuke rows with n/as

df = df.drop('pub_rec_bankruptcies',axis=1)
df = df.dropna()

In [7]:
df.shape

(37675, 22)

What do our non-numerical columns look like?

In [8]:
df.select_dtypes('object').head().iloc[0]

term                     36 months
int_rate                    10.65%
emp_length               10+ years
home_ownership                RENT
verification_status       Verified
purpose                credit_card
title                     Computer
addr_state                      AZ
earliest_cr_line          Jan-1985
revol_util                   83.7%
last_credit_pull_d        Jun-2016
Name: 0, dtype: object

Let's take a closer look at the distinct categories in these columns. We will want to avoid encoding those fields with many discrete values as that will complicate our model and slow down processing speed.

In [9]:
cols = ['home_ownership', 'verification_status', 'emp_length', 'term', 'addr_state','purpose','title']

for c in cols:
        print(df[c].value_counts())

RENT        18112
MORTGAGE    16686
OWN          2778
OTHER          96
NONE            3
Name: home_ownership, dtype: int64
Not Verified       16281
Verified           11856
Source Verified     9538
Name: verification_status, dtype: int64
10+ years    8545
< 1 year     4513
2 years      4303
3 years      4022
4 years      3353
5 years      3202
1 year       3176
6 years      2177
7 years      1714
8 years      1442
9 years      1228
Name: emp_length, dtype: int64
 36 months    28234
 60 months     9441
Name: term, dtype: int64
CA    6776
NY    3614
FL    2704
TX    2613
NJ    1776
IL    1447
PA    1442
VA    1347
GA    1323
MA    1272
OH    1149
MD    1008
AZ     807
WA     788
CO     748
NC     729
CT     711
MI     678
MO     648
MN     581
NV     466
SC     454
WI     427
OR     422
AL     420
LA     420
KY     311
OK     285
UT     249
KS     249
AR     229
DC     209
RI     194
NM     180
WV     164
HI     162
NH     157
DE     110
MT      77
WY      76
AK      76
SD      60
VT  

- We ought to convert ```emp_length``` to numeric becuase they have ordering (4 years > 2 years)
- ```home_ownership```, ```verification_status```, ```emp_length```, and ```term``` columns contain few discrete categorical values. We will encode these as dummy variables and hold on to them
- ```addr_state``` and ```title``` are information overloads so we will toss those
- Lastly we will convert ```int_rate``` and ```revol_util``` to float

In [10]:
mapping = {         
    "emp_length": {
        "10+ years": 10,
        "9 years": 9,
        "8 years": 8,
        "7 years": 7,
        "6 years": 6,
        "5 years": 5,
        "4 years": 4,
        "3 years": 3,
        "2 years": 2,
        "1 year": 1,
        "< 1 year": 0,
        "n/a": 0
    }
}

df = df.replace(mapping)

df = df.drop(['title','addr_state','last_credit_pull_d','earliest_cr_line'],axis=1)

df['int_rate'] = df['int_rate'].str.rstrip('%').astype('float')
df['revol_util'] = df['revol_util'].str.rstrip('%').astype('float')

In [11]:
df.head()

,loan_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc
0,5000.0,36 months,10.65,162.87,10,RENT,24000.0,Verified,1,credit_card,27.65,0.0,1.0,3.0,0.0,13648.0,83.7,9.0
1,2500.0,60 months,15.27,59.83,0,RENT,30000.0,Source Verified,0,car,1.00,0.0,5.0,3.0,0.0,1687.0,9.4,4.0
2,2400.0,36 months,15.96,84.33,10,RENT,12252.0,Not Verified,1,small_business,8.72,0.0,2.0,2.0,0.0,2956.0,98.5,10.0
3,10000.0,36 months,13.49,339.31,10,RENT,49200.0,Source Verified,1,other,20.00,0.0,1.0,10.0,0.0,5598.0,21.0,37.0
4,5000.0,36 months,7.90,156.46,3,RENT,36000.0,Source Verified,1,wedding,11.20,0.0,3.0,9.0,0.0,7963.0,28.3,12.0


### Dummy Variables

Let's encode the home_ownership, verification_status, purpose, and term columns as dummy variables so we can use them in our model.

In [12]:
d = pd.get_dummies(df[['home_ownership', 'verification_status', 'purpose', 'term']])

df = pd.concat([df,d],axis=1)
df = df.drop(['home_ownership', 'verification_status', 'purpose', 'term'],axis=1)

In [13]:
df.shape

(37675, 38)

Our dataframe is now ready to start training for machine learning models.

In [14]:
df.to_csv('loans_final.csv',index=False)